# LlamaIndex와 AgentCore 메모리 - 학술 연구 어시스턴트 (단기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 학술 연구 어시스턴트를 만드는 방법을 보여줍니다. 단일 연구 세션 내에서의 **단기 메모리** 지속성에 초점을 맞추며, 대화 전반에 걸쳐 논문, 연구 결과 및 연구 맥락을 기억할 수 있도록 합니다.

## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형 메모리                                                               |
| 에이전트 사용 사례  | 학술 연구 어시스턴트                                                             |
| 에이전트 프레임워크 | LlamaIndex                                                                       |
| LLM 모델            | Anthropic Claude 3.7 Sonnet                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, LlamaIndex 에이전트, 연구 도구                            |
| 예제 난이도         | 초급                                                                             |

학습 내용:
- 연구 데이터 지속성을 위한 AgentCore 메모리 생성
- LlamaIndex 네이티브 메모리 통합 사용
- 논문 분석을 위한 연구 전용 도구 구축
- 단일 세션 내에서 연구 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 컨텍스트

이 예제에서는 연구자가 단일 연구 세션 내에서 논문, 연구 결과 및 연구 주제를 추적하는 데 도움을 주는 "학술 연구 어시스턴트"를 만듭니다. 이 어시스턴트는 AgentCore 메모리를 사용하여 대화 전반에 걸쳐 검토한 논문, 발견한 주요 결과 및 연구 진행 상황에 대한 컨텍스트를 유지합니다.

## 아키텍처 개요

![LlamaIndex AgentCore 단기 메모리 아키텍처](LlamaIndex-AgentCore-STM-Arch.png)

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 의존성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore 메모리 구성

연구 어시스턴트를 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# AgentCore 메모리 리소스 생성
region = os.getenv('AWS_REGION', 'us-east-1')
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f'AcademicResearchShortTerm_{int(datetime.now().timestamp())}',
        description='Academic research assistant short-term memory for single session context',
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )
    memory_id = response['id']
    print(f"AgentCore 메모리 생성 완료: {memory_id}")
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 연구 도구 구현

학술 연구 작업을 위한 전문 도구를 정의합니다:

In [ ]:
def save_paper_summary(title: str, authors: str, key_findings: str) -> str:
    """Save a research paper summary with title, authors, and key findings"""
    print(f"논문 저장 완료: {title} by {authors}")
    return f"Successfully saved paper summary for '{title}'"

def track_research_topic(topic: str, status: str) -> str:
    """Track research topic progress with current status"""
    print(f"연구 주제 추적 중: {topic} (상태: {status})")
    return f"Now tracking research topic: {topic} with status {status}"

def save_research_finding(finding: str, confidence: str) -> str:
    """Save a research finding with confidence level"""
    print(f"연구 결과 저장 완료 (신뢰도: {confidence})")
    return f"Saved research finding with {confidence} confidence level"

# 에이전트용 도구 객체 생성
research_tools = [
    FunctionTool.from_defaults(fn=save_paper_summary),
    FunctionTool.from_defaults(fn=track_research_topic),
    FunctionTool.from_defaults(fn=save_research_finding)
]

## 4단계: LlamaIndex 에이전트 구현

단기 메모리 컨텍스트를 가진 연구 어시스턴트 에이전트를 생성합니다:

In [ ]:
# 단기 메모리 구성 (단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="academic-researcher",
    memory_id=memory_id,
    session_id="research-session-today",  # 전체에서 동일한 세션
    namespace="/academic-research/"
)

# AgentCore 메모리 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 연구 어시스턴트 에이전트 생성
research_agent = FunctionAgent(
    tools=research_tools,
    llm=llm,
    verbose=True
)

print("단기 메모리가 적용된 학술 연구 어시스턴트 준비 완료!")

## 5단계: 단기 메모리 기능 테스트

포괄적인 연구 세션을 통해 연구 어시스턴트의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 세션 초기화

In [ ]:
# 상세한 컨텍스트로 연구 세션 초기화
# (MIT 컴퓨터과학과 Sarah Smith 박사로서, '헬스케어에서의 머신러닝 응용' 연구를 시작하고 문헌 검토 상태로 추적 요청)
response = await research_agent.run(
    "I'm Dr. Sarah Smith from MIT's Computer Science Department, starting research on 'Machine Learning in Healthcare Applications'. "
    "Track this topic with status 'Literature Review'.",
    memory=agentcore_memory
)

print("세션 초기화:")
print(response)

### 테스트 2: 연구 논문 추가

In [ ]:
# 상세한 지표와 함께 첫 번째 논문 추가
# (Zhang et al.의 '의료 영상 분석을 위한 딥러닝' 논문 저장 - 흉부 X선 진단에서 CNN이 95.2% 정확도 달성)
response = await research_agent.run(
    "Save paper: 'Deep Learning for Medical Image Analysis' by Zhang et al. "
    "Key findings: CNNs achieve 95.2% accuracy in chest X-ray diagnosis, 12% improvement over radiologists, "
    "trained on 100,000 images with 0.03 false positive rate.",
    memory=agentcore_memory
)

print("논문 1 추가:")
print(response)

In [ ]:
# 대조적인 결과를 가진 두 번째 논문 추가
# (Johnson et al.의 '의료 NLP에서의 트랜스포머' 논문 저장 - BERT 모델이 임상 노트 분류에서 89.1% F1-스코어 달성)
response = await research_agent.run(
    "Save paper: 'Transformers in Medical NLP' by Johnson et al. "
    "Key findings: BERT models achieve 89.1% F1-score in clinical note classification, "
    "struggle with rare diseases (<70% accuracy), excel at symptom extraction (94% precision).",
    memory=agentcore_memory
)

print("논문 2 추가:")
print(response)

### 테스트 3: 신원 및 컨텍스트 회상

In [ ]:
# 신원 및 연구 컨텍스트 회상 테스트
# (이름, 소속 기관, 현재 연구 분야 질문)
response = await research_agent.run(
    "What's my name, institution, and current research focus?",
    memory=agentcore_memory
)

print("신원 회상 테스트:")
print(response)
print("\n예상 결과: Dr. Sarah Smith, MIT, Machine Learning in Healthcare")

### 테스트 4: 상세 지표 회상

In [ ]:
# 특정 지표 회상 테스트
# (검토한 논문들에서 언급된 정확한 정확도 퍼센티지와 CNN vs 트랜스포머 저자 질문)
response = await research_agent.run(
    "What were the exact accuracy percentages mentioned in the papers I reviewed? "
    "Which authors wrote about CNNs vs Transformers?",
    memory=agentcore_memory
)

print("상세 지표 회상:")
print(response)
print("\n예상 결과: Zhang et al - CNN 95.2%, Johnson et al - BERT 89.1%")

### 테스트 5: 문맥 추론

In [ ]:
# 문맥 이해 및 추론 테스트
# (검토한 논문을 기반으로 흉부 X선 분석 vs 임상 노트 분석에 어떤 접근법이 더 나은지 질문)
response = await research_agent.run(
    "Based on the papers I've reviewed, which approach would be better for analyzing "
    "chest X-rays vs clinical notes? Explain your reasoning.",
    memory=agentcore_memory
)

print("문맥 추론 테스트:")
print(response)
print("\n예상 결과: X선에는 CNN(Zhang 논문), 임상 노트에는 트랜스포머(Johnson 논문)")

### 테스트 6: 연구 결과 종합

In [ ]:
# 종합된 연구 결과 추가
# (Zhang의 CNN 결과(95.2%)와 Johnson의 트랜스포머 결과(89.1%)를 기반으로, 딥러닝 모델이 헬스케어에서 85% 이상 정확도를 달성한다는 결론을 높은 신뢰도로 저장)
response = await research_agent.run(
    "Based on Zhang's CNN results (95.2% accuracy) and Johnson's Transformer results (89.1% F1-score), "
    "I conclude that deep learning models consistently achieve >85% accuracy in healthcare tasks. "
    "This finding has high confidence. Save it.",
    memory=agentcore_memory
)

print("연구 결과 종합:")
print(response)

### 테스트 7: 교차 참조 기능

In [ ]:
# 연구 결과와 논문 간의 교차 참조 테스트
# (85% 이상 정확도에 대한 연구 결과가 Zhang과 Johnson의 구체적 결과와 어떻게 관련되는지 질문)
response = await research_agent.run(
    "How does my research finding about >85% accuracy relate to the specific results "
    "from Zhang and Johnson? What evidence supports this conclusion?",
    memory=agentcore_memory
)

print("교차 참조 테스트:")
print(response)
print("\n예상 결과: Zhang 95.2%와 Johnson 89.1%를 근거 자료로 참조")

### 테스트 8: 실용적 응용 시나리오

In [ ]:
# 축적된 지식의 실용적 응용 테스트
# (헬스케어 AI 연구를 위한 연구비 제안서를 작성 중이며, 딥러닝 효과에 대한 근거 요청)
response = await research_agent.run(
    "I'm writing a grant proposal for healthcare AI research. What evidence can I cite "
    "about deep learning effectiveness? Include specific numbers and authors.",
    memory=agentcore_memory
)

print("연구비 제안서 지원:")
print(response)
print("\n예상 결과: Zhang 95.2%, Johnson 89.1%, 종합 결과를 포함한 포괄적 요약")

## 6단계: 세션 경계 테스트

다른 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다:

In [ ]:
# 다른 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="academic-researcher",
    memory_id=memory_id,
    session_id="different-research-session",  # 다른 세션 ID
    namespace="/academic-research/"
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
# (어떤 연구를 하고 있었는지, 어떤 정확도 수치를 발견했는지 질문)
response = await research_agent.run(
    "What research have I been working on? What specific accuracy numbers did I find?",
    memory=new_session_memory
)

print("세션 경계 테스트 (다른 세션):")
print(response)
print("\n예상 결과: 이전 세션에서의 회상이 제한적이거나 없음 (단기 메모리 경계)")

In [ ]:
# 원래 세션으로 돌아가서 지속성 확인
# (원래 세션으로 돌아와서 Zhang과 Johnson의 정확도 수치 다시 질문)
response = await research_agent.run(
    "Now back in my original session - what were the accuracy numbers from Zhang and Johnson again?",
    memory=agentcore_memory  # 원래 세션 메모리
)

print("원래 세션 복귀:")
print(response)
print("\n예상 결과: Zhang 95.2%, Johnson 89.1% 전체 회상")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀들을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 세션 초반 정보를 회상할 수 있는지 확인"""
        # 실질적인 응답인지 확인 (단순한 "모르겠습니다"가 아닌지)
        has_content = len(response) > 50
        # 메모리 지표 확인
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내에서 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        # 연결 언어 찾기
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동합니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하며, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 중요한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - 에이전트가 논의된 내용을 회상할 수 있는가?
response1 = await research_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리 - 에이전트가 컨텍스트를 유지하는가?
response2 = await research_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 기능 - 이전 컨텍스트와 연결할 수 있는가?
response3 = await research_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 노트북에서 다음을 시연했습니다:

- **단기 메모리 통합**: 세션 범위 지속성을 위해 LlamaIndex와 AgentCore 메모리 사용

- **연구 전용 도구**: 논문 요약, 주제 추적 및 연구 결과 저장

- **문맥 대화**: 어시스턴트가 세션 내에서 상세한 정보를 기억

- **교차 참조 기능**: 여러 논문과 상호작용에 걸쳐 연구 결과 연결

- **세션 경계**: 서로 다른 대화 세션 간의 메모리 격리

- **실용적 응용**: 연구비 제안서 지원 및 연구 종합

학술 연구 어시스턴트는 단기 메모리가 어떻게 단일 연구 세션 내에서 자연스럽고 문맥에 맞는 대화를 가능하게 하면서 서로 다른 대화 스레드 간에 명확한 경계를 유지하는지 보여줍니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다:

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")